In [1]:
import os
import pandas as pd
from tqdm.notebook import tqdm

from utilities import get_date, walk_through_files, get_subzone_name_from_dir

pd.set_option("display.max_rows", 500)

RECORDS_PATH = os.path.join("raw-data", "sg_subzones_devices", "sg_subzones_devices")
SAVEPATH = os.path.join("intermediate-files", "devicecount-subzone-day.csv")

In [2]:
ix = 0
dict_placeholder = {}


for filepath in tqdm(walk_through_files(RECORDS_PATH, "csv.gz", fullpath=True)):

    # parse file name to get date and loc
    try:
        subzone = get_subzone_name_from_dir(filepath)
        date = get_date(filepath)
    except ValueError:
        print(filepath)

    # read into df
    _df = pd.read_csv(
        filepath,
        compression="gzip",
        names=["id", "os"],
        dtype={"id": "object", "os": "int8"},
    )

    android_count = len(_df[_df["os"] == 0])
    ios_count = len(_df[_df["os"] == 1])

    # init list and add to dict
    dict_placeholder[ix] = [subzone, date, android_count, ios_count]

    ix += 1

0it [00:00, ?it/s]

In [5]:
df = (
    pd.DataFrame.from_dict(
        dict_placeholder,
        orient="index",
        columns=["subzone", "date", "android_count", "ios_count"],
    )
    .astype(
        {
            "subzone": "object",
            "date": "datetime64[ns]",
            "android_count": "int",
            "ios_count": "int",
        }
    )
    .drop_duplicates(["subzone", "date"], ignore_index=True)
    .sort_values(["date", "subzone"], ignore_index=True)
)

df.head(10)

,subzone,date,android_count,ios_count
0,admiralty,2020-01-01,63,10
1,airport road,2020-01-01,54,3
2,alexandra hill,2020-01-01,1080,203
3,alexandra north,2020-01-01,362,74
4,aljunied,2020-01-01,5687,1033
5,anak bukit,2020-01-01,1296,310
6,anchorvale,2020-01-01,1894,380
7,ang mo kio town centre,2020-01-01,1200,176
8,anson,2020-01-01,265,77
9,balestier,2020-01-01,3543,1289


In [17]:
(
    df
    .groupby("subzone")[["android_count", "ios_count"]]
    .sum()
    .assign(ios_android_ratio=lambda df_: df_["ios_count"] / df_["android_count"])
    .sort_values(["ios_android_ratio"], ascending=False)
)

,android_count,ios_count,ios_android_ratio
subzone,,,
pulau punggol timor,1,2,2.000000
pulau punggol barat,8,8,1.000000
istana negara,82996,56797,0.684334
holland road,134189,80735,0.601651
somerset,134044,77879,0.580996
kovan,574231,280308,0.488145
tiong bahru,126356,61290,0.485058
coronation road,66740,31523,0.472325
upper thomson,353564,150373,0.425306


In [ ]:
df.to_csv(SAVEPATH, index=False)